# exp_18 - Tier 2A Phase B: FINETUNE the masked-recon encoder (track_c, Session 3)

Plan: docs/track_3_rl_reuse/session_plan.md §5 Session 3 step 1. Runs on **local AND Kaggle**.

**Idea (plain):** take the encoder pretrained by exp_18 (masked reconstruction), drop the recon
decoder, attach a FRESH 1-output rain head, and finetune the WHOLE model (encoder unfrozen from
epoch 1) on next-day rain. Compare to the Tier 0b baseline (exp_16, 15-feat) on RMSE + train-val gap.

**This is Tier 2A (two-stage: pretrain -> finetune).** Not the joint model (that is exp_19 / Tier 2B).
Single seed 42; multi-seed only at Session 5.

In [1]:
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import glob
from torch.utils.data import DataLoader

In [2]:
# --- environment bootstrap: identical notebook runs on BOTH local and Kaggle ---
if os.path.exists('/kaggle/working'):                       # ---- Kaggle ----
    REPO = '/kaggle/working/Dissertation'
    if not os.path.exists(REPO):
        os.system('git clone --depth 1 https://github.com/itsRkg/Dissertation.git ' + REPO)
    os.system('pip install -q torch_geometric')   # Kaggle lacks PyG (provides GATConv); local already has it
    KAGGLE_DATA_SLUG = 'REPLACE_WITH_YOUR_KAGGLE_DATASET_SLUG'   # e.g. 'itsrkg/dissertation-era5'
else:                                                        # ---- local ----
    REPO = 'C:/Users/rishe/Dissertation'
    KAGGLE_DATA_SLUG = None
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('repo on path:', REPO)

repo on path: C:/Users/rishe/Dissertation


In [4]:
from utils.data_utils.data_helper_utils import (
    load_pivots, get_lat_lon_aligned, build_edge_index_radius, scale_pivots, temporal_split)
from utils.data_utils.dataset_files.gnn_dataset import (
    build_feature_tensor, build_time_features, add_time_features, SpatioTemporalDataset)
from models.gat_gru import GAT_GRU_Model
from utils.train_utils import train_model, evaluate
from utils.metric_utils.metrics import rmse, mae, bias, nrmse
from utils.metric_utils.extreme_skill import extreme_skill_table
from utils.metric_utils.embedding_diag import embedding_report, last_gat_embeddings
from utils.env_paths import get_paths

In [5]:
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# finetune hyperparams = exp_16/exp_13 (lr 1e-3, batch 32, 30 ep, MSELoss); 15-feat (anchored to Tier 0b)
CFG = dict(epochs=30, lr=1e-3, batch_size=32, hidden_dim=64, heads=4, seq_len=7, horizon=1, in_channels=15)
P = get_paths(kaggle_data_slug=KAGGLE_DATA_SLUG)
if P['is_kaggle']:
    hits = glob.glob('/kaggle/input/**/rain_pivot.parquet', recursive=True)
    assert hits, 'rain_pivot.parquet not found under /kaggle/input - attach the dataset'
    DATA = os.path.dirname(hits[0]); P['pivot_dir'] = DATA + '/'
    P['coords_csv'] = glob.glob('/kaggle/input/**/wb_station_coords.csv', recursive=True)[0]
    P['elev_csv']   = glob.glob('/kaggle/input/**/wb_station_elevation.csv', recursive=True)[0]
# locate the Session-2 pretext checkpoint: local repo first, then anywhere under repo / Kaggle input
CKPT = P['models_dir'] + '/exp_18_gat_gru_mae_pretrain/pretext/pretext_best.pt'
if not os.path.exists(CKPT):
    cand = glob.glob(P['repo'] + '/**/pretext_best.pt', recursive=True) + glob.glob('/kaggle/input/**/pretext_best.pt', recursive=True)
    assert cand, 'pretext_best.pt not found - run exp_18 pretrain (Session 2) first, or attach it on Kaggle'
    CKPT = cand[0]
PIVOT_PATH = P['pivot_dir']
RES_DIR  = P['results_dir'] + '/exp_18_gat_gru_mae_pretrain/finetune/seed_' + str(SEED)
SAVE_DIR = P['models_dir']  + '/exp_18_gat_gru_mae_pretrain/finetune/seed_' + str(SEED)
LOG_DIR  = P['logs_dir']    + '/exp_18_gat_gru_mae_pretrain/finetune/seed_' + str(SEED)
for d in (RES_DIR, SAVE_DIR, LOG_DIR): os.makedirs(d, exist_ok=True)
print(('KAGGLE' if P['is_kaggle'] else 'LOCAL'), '| data', PIVOT_PATH)
print('pretext ckpt:', CKPT)
print('device', device, '| cfg', CFG)

LOCAL | data C:\Users\rishe\Dissertation\data\era5_pivot_data/
pretext ckpt: C:\Users\rishe\Dissertation\experiments\saved_models/exp_18_gat_gru_mae_pretrain/pretext/pretext_best.pt
device cuda | cfg {'epochs': 30, 'lr': 0.001, 'batch_size': 32, 'hidden_dim': 64, 'heads': 4, 'seq_len': 7, 'horizon': 1, 'in_channels': 15}


In [6]:
# ---- data + graph + scaler fix + 15-feat tensor (IDENTICAL recipe to exp_16 tier0b / exp_18) ----
pivots = load_pivots(PIVOT_PATH)
station_df = pd.read_csv(P['coords_csv']).rename(columns={'latitude':'lat','longitude':'lon'})
elev_df = pd.read_csv(P['elev_csv'])
lat, lon = get_lat_lon_aligned(pivots['rain'], station_df)
edge_index = build_edge_index_radius(lat, lon, threshold_km=100).to(device)
N = pivots['rain'].shape[1]
dates = pivots['rain'].index
T = len(dates); train_end_date = dates[int(0.7*T)-1]
print('train_end_date', train_end_date)
scaled_pivots, scalers = scale_pivots(pivots, train_end=train_end_date)
mu, sigma = scalers['rain']
X_dyn, feature_order = build_feature_tensor(scaled_pivots, use_latent=True)   # 6 physical, rain=idx0
X12 = add_time_features(X_dyn, build_time_features(dates))                    # +6 cyclic time
elev_map = dict(zip(elev_df['station_id'].astype(str), elev_df['elevation_m'].astype(float)))
assert not [c for c in pivots['rain'].columns if str(c) not in elev_map], 'elevation missing'
elev = np.array([elev_map[str(c)] for c in pivots['rain'].columns], dtype=float)
def _z(a):
    a = np.asarray(a, float); s = a.std(); return (a - a.mean())/(s if s else 1.0)
statics = np.stack([_z(lat), _z(lon), _z(elev)], axis=-1)
X15 = np.concatenate([X12, np.broadcast_to(statics[None,:,:], (X12.shape[0], N, 3))], axis=-1)
assert np.allclose(X15[...,0], X12[...,0]) and X15.shape[-1] == 15   # rain stays feature 0
print('X15', X15.shape, '| rain scaler mu=%.4f sigma=%.4f' % (mu, sigma), '| feat0 =', feature_order[0])

train_end_date 2007-10-19 00:00:00
X15 (19723, 291, 15) | rain scaler mu=5.3051 sigma=12.5862 | feat0 = rain


In [7]:
Xtr, Xva, Xte = temporal_split(X15, dates)   # same 70/15/15 chronological split as exp_16
train_loader = DataLoader(SpatioTemporalDataset(Xtr, seq_len=CFG['seq_len'], horizon=CFG['horizon']), batch_size=CFG['batch_size'], shuffle=True)
val_loader   = DataLoader(SpatioTemporalDataset(Xva, seq_len=CFG['seq_len'], horizon=CFG['horizon']), batch_size=CFG['batch_size'])
test_loader  = DataLoader(SpatioTemporalDataset(Xte, seq_len=CFG['seq_len'], horizon=CFG['horizon']), batch_size=CFG['batch_size'])
print('splits', Xtr.shape, Xva.shape, Xte.shape)

splits (13806, 291, 15) (2958, 291, 15) (2959, 291, 15)


In [8]:
# ---- build forecast model, TRANSFER the pretrained encoder (gat + gru), fresh rain head ----
model = GAT_GRU_Model(in_channels=CFG['in_channels'], hidden_dim=CFG['hidden_dim'], heads=CFG['heads'])
pre_sd = torch.load(CKPT, map_location='cpu')   # plain state_dict of GAT_GRU_Pretrain: gat.* / gru.* / decoder.*
gat_sd = {k[len('gat.'):]: v for k, v in pre_sd.items() if k.startswith('gat.')}
gru_sd = {k[len('gru.'):]: v for k, v in pre_sd.items() if k.startswith('gru.')}
assert gat_sd and gru_sd, 'checkpoint missing gat./gru. keys; got ' + str(list(pre_sd)[:6])
model.gat.load_state_dict(gat_sd)   # strict (default): raises if keys/shapes mismatch
model.gru.load_state_dict(gru_sd)
print('transferred pretrained encoder: gat tensors', len(gat_sd), '| gru tensors', len(gru_sd), '| fc head = fresh/random')
# NOTE: train_model builds Adam over ALL params (gat+gru+fc) -> encoder is FULLY UNFROZEN from epoch 1
#       (mirrors the deployed s14_unfrozen_encoder; session_plan §5 Session 3).

transferred pretrained encoder: gat tensors 4 | gru tensors 4 | fc head = fresh/random


C:\Users\rishe\AppData\Local\Temp\ipykernel_59760\3781518187.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pre_sd = torch.load(CKPT, map_location='cpu')   # plain stat

In [9]:
def real_metrics(yt, yp):
    return {'RMSE': rmse(yt,yp), 'MAE': mae(yt,yp), 'Bias': bias(yt,yp), 'NRMSE': nrmse(yt,yp)}
SEASONS = {'monsoon':[6,7,8,9], 'non_monsoon':[1,2,3,4,5,10,11,12]}

In [10]:
# ---- finetune (forecast-only MSE) then evaluate from best-val checkpoint ----
exp_name = 'exp_18_finetune_seed' + str(SEED)
model = train_model(train_loader, val_loader, model, edge_index, device,
                    epochs=CFG['epochs'], lr=CFG['lr'], criterion=nn.MSELoss(),
                    save_dir=SAVE_DIR, log_dir=LOG_DIR, experiment_name=exp_name)
model.load_state_dict(torch.load(SAVE_DIR + '/' + exp_name + '_best.pt', map_location=device))
crit = nn.MSELoss()
_, preds, targets = evaluate(model, test_loader, crit, edge_index, device)
train_loss, _, _ = evaluate(model, train_loader, crit, edge_index, device)
val_loss, _, _   = evaluate(model, val_loader,  crit, edge_index, device)
gap = float(val_loss - train_loss)
preds_real = preds*sigma + mu; targets_real = targets*sigma + mu

2026-05-30 16:48:24 | INFO | Starting GCN+GRU training
2026-05-30 16:48:24 | INFO | Device: cuda
2026-05-30 16:48:24 | INFO | Experiment Config:
2026-05-30 16:48:24 | INFO | LR: 0.001
2026-05-30 16:48:24 | INFO | Epochs: 30
2026-05-30 16:48:24 | INFO | Criterion: MSELoss()


Train:   0%|          | 0/432 [00:00<?, ?it/s]

RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ---- real-space metrics + full save layout (IDENTICAL to exp_16) ----
np.savez_compressed(RES_DIR + '/test_predictions_real.npz', obs=targets_real, pred=preds_real)
ov = real_metrics(targets_real.reshape(-1), preds_real.reshape(-1))
pd.DataFrame([ov]).to_csv(RES_DIR + '/overall_metrics.csv', index=False)
dt = dates[-len(Xte):]; months = dt.month.values[CFG['seq_len']:]
months_flat = np.repeat(months[:,None], preds.shape[1], axis=1).reshape(-1)
seas = []
for sname, sm in SEASONS.items():
    m = np.isin(months_flat, sm)
    r = real_metrics(targets_real.reshape(-1)[m], preds_real.reshape(-1)[m]); r['season'] = sname; seas.append(r)
pd.DataFrame(seas).to_csv(RES_DIR + '/seasonal_metrics_real.csv', index=False)
ids = pivots['rain'].columns.tolist()
st = [dict(real_metrics(targets_real[:,i], preds_real[:,i]), station_id=ids[i]) for i in range(preds.shape[1])]
pd.DataFrame(st).to_csv(RES_DIR + '/station_metrics_real.csv', index=False)
ext = extreme_skill_table(targets_real.reshape(-1), preds_real.reshape(-1))
ext.to_csv(RES_DIR + '/extreme_skill_real.csv', index=False)
e645 = ext[np.isclose(ext['threshold_mm'], 64.5)].iloc[0]
xb = next(iter(test_loader)); xb = xb[0] if isinstance(xb, (list, tuple)) else xb
emb = last_gat_embeddings(model, xb, edge_index, device)
ei_np = edge_index.detach().cpu().numpy()
reps = [embedding_report(emb[b].cpu().numpy(), ei_np) for b in range(emb.shape[0])]
edrep = {k: float(np.mean([r[k] for r in reps])) for k in reps[0]}
pd.DataFrame([edrep]).to_csv(RES_DIR + '/embedding_diag.csv', index=False)
print('RMSE %.3f | MAE %.3f | Bias %.3f | NRMSE %.3f | gap %.4f | CSI@64.5 %.3f' % (
      ov['RMSE'], ov['MAE'], ov['Bias'], ov['NRMSE'], gap, float(e645['CSI'])))

In [ ]:
# ---- one-row summary + comparison vs the Tier 0b anchor (exp_16, 15-feat) ----
row = dict(config='ssl_pretrain_finetune', tier='2A', in_channels=15, seed=SEED,
           RMSE=ov['RMSE'], MAE=ov['MAE'], Bias=ov['Bias'], NRMSE=ov['NRMSE'],
           train_loss=float(train_loss), val_loss=float(val_loss), train_val_gap=gap,
           CSI_64p5=float(e645['CSI']), POD_64p5=float(e645['POD']), n_events_64p5=int(e645['n_events']),
           dirichlet_norm=edrep['dirichlet_energy_norm'], MAD_cosine=edrep['MAD_cosine'], eff_rank=edrep['effective_rank'])
pd.DataFrame([row]).to_csv(RES_DIR + '/finetune_summary.csv', index=False)
anchor = P['results_dir'] + '/exp_16_gat_gru_baseline_seeds/tier0b/seed_42/overall_metrics.csv'
if os.path.exists(anchor):
    b = float(pd.read_csv(anchor).iloc[0]['RMSE']); d = ov['RMSE'] - b
    verdict = 'MULTI-SEED CANDIDATE (>=0.2 better)' if d <= -0.2 else ('worse than 0b' if d > 0 else 'marginal')
    print('Tier 0b RMSE %.3f  ->  Tier 2A finetune RMSE %.3f  | delta %+.3f  -> %s' % (b, ov['RMSE'], d, verdict))
    print('reroute gate (session_plan §7): flag for Session-5 multi-seed if delta <= -0.2 RMSE OR gap shrinks >= 0.3')
else:
    print('tier0b anchor not found at', anchor, '- compare manually to RMSE 8.777 (exp_16 tier0b, seed 42)')

## How to read

- **Headline:** finetune RMSE vs **Tier 0b 8.777** (matched 15-feat anchor, NOT exp_13 8.79). Also watch `train_val_gap` (baseline gap is ~0, so a gap reduction is the other way SSL can win).
- `extreme_skill_real.csv` = PRIVATE limitation diagnostic (expect low CSI@64.5). Never the headline (session_plan §0).
- `embedding_diag.csv` on the finetuned encoder - compare to the pretext diag (Dir 0.089 / MAD 0.43) and Tier 0b (0.042 / 0.246).
- Single-seed = exploratory. Nothing final until Session 5 multi-seed. Honest expectation: a marginal/null lift is plausible and still a defensible chapter (documented null methodology result).